In [1]:
from hdfs import InsecureClient
import pandas as pd

## 1. Load penguins.csv from HDFS into Pandas

In [2]:
client = InsecureClient('http://127.0.0.1:50070', user='maria_dev')

filepath = '/user/maria_dev/datasets/penguins.csv'

with client.read(filepath) as reader:
    df = pd.read_csv(reader)

print(df.head())


ConnectionError: HTTPConnectionPool(host='sandbox-hdp.hortonworks.com', port=50075): Max retries exceeded with url: /webhdfs/v1/user/maria_dev/datasets/penguins.csv?op=OPEN&user.name=maria_dev&namenoderpcaddress=sandbox-hdp.hortonworks.com:8020&offset=0 (Caused by NameResolutionError("HTTPConnection(host='sandbox-hdp.hortonworks.com', port=50075): Failed to resolve 'sandbox-hdp.hortonworks.com' ([Errno 11001] getaddrinfo failed)"))

In [ ]:
print('Shape:', df.shape)
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
print('Missing values per column:')
df.isnull().sum()

## 2. Clean and preprocess

In [ ]:
clean = df.copy()

# Drop rows missing ALL body measurements (no useful data)
measure_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
clean = clean.dropna(subset=measure_cols, how='all')

# Fill remaining numeric gaps with the species median
for c in measure_cols:
    clean[c] = clean.groupby('species')[c].transform(lambda s: s.fillna(s.median()))

# Fill missing sex with 'unknown'
clean['sex'] = clean['sex'].fillna('unknown')

# Convert body mass to kilograms
clean['body_mass_kg'] = (clean['body_mass_g'] / 1000).round(3)
clean = clean.drop(columns=['body_mass_g'])

# Tidy types
clean['flipper_length_mm'] = clean['flipper_length_mm'].astype(int)

print('Shape after cleaning:', clean.shape)
clean.isnull().sum()

In [ ]:
clean.head()

## 3. Upload cleaned data to HDFS

In [ ]:
client.makedirs('/user/maria_dev/datasets/clean')
with client.write('/user/maria_dev/datasets/clean/penguins_clean.csv',
                  encoding='utf-8', overwrite=True) as writer:
    clean.to_csv(writer, index=False)

client.list('/user/maria_dev/datasets/clean')

In [ ]:
# Verify by reading it back
with client.read('/user/maria_dev/datasets/clean/penguins_clean.csv', encoding='utf-8') as r:
    check = pd.read_csv(r)
print(check.shape)
check.head()